In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings("ignore")

In [5]:
data = pd.read_csv("patient_data_processed.csv")

X = data.drop(columns=['Patient', 'Stages'])
y = data['Stages']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
models = {

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=9))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=10,
        random_state=42
    ),

    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel='linear'))
    ]),

    "RBF SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel='rbf'))
    ]),

    "Ridge Classifier": Pipeline([
        ("scaler", StandardScaler()),
        ("model", RidgeClassifier())
    ]),

    "Gaussian Naive Bayes": GaussianNB()
}

In [8]:
results = []

for name, model in models.items():

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    cv_scores = cross_val_score(model, X, y, cv=5)
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1,
        cv_mean,
        cv_std
    ])

In [9]:
comparison_df = pd.DataFrame(results, columns=[
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "CV Mean Accuracy",
    "CV Std Dev"
])

comparison_df = comparison_df.sort_values(by="Accuracy", ascending=False)

comparison_df

,Model,Accuracy,Precision,Recall,F1-Score,CV Mean Accuracy,CV Std Dev
2,Random Forest,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
3,Decision Tree,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
5,RBF SVM,1.000000,1.000000,1.000000,1.000000,0.982466,0.025560
1,KNN,0.994521,0.994679,0.994521,0.994540,0.920000,0.066872
0,Logistic Regression,0.972603,0.977326,0.972603,0.973299,0.911233,0.081598
4,Linear SVM,0.972603,0.977326,0.972603,0.973299,0.934247,0.065019
6,Ridge Classifier,0.953425,0.954545,0.953425,0.953836,0.899178,0.087232
7,Gaussian Naive Bayes,0.884932,0.938630,0.884932,0.890238,0.894795,0.135395
